# PATHQ — Week 3: Classical GNN Baseline (ABMIL)

**Goal:** Train classical GNN+ABMIL. This is your paper Table 1 Row 1.

**By end of this notebook:**
- AUC > 0.90 on PatchCamelyon
- CAMELYON16 baseline AUC recorded
- Attention maps visualised (XAI Layer 2 preview)
- Results saved to outputs/baseline_results.json

In [1]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, f1_score
import wandb, warnings, sys, json
warnings.filterwarnings('ignore')
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as PyGLoader
from torch_geometric.nn import GCNConv
from scipy.spatial import KDTree

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42); np.random.seed(42)

# Auto-detect the project root that actually contains extracted data\ndef _pick_project_root():\n    candidates = [Path.cwd(), Path.cwd() / 'notebooks', Path.cwd().parent]\n    best_root, best_score = candidates[0], -1\n    for root in candidates:\n        features = len(list((root / 'data' / 'features').glob('*.pt')))\n        normal   = len(list((root / 'data' / 'camelyon16' / 'normal').glob('*.tif'))) + len(list((root / 'data' / 'camelyon16' / 'normal').glob('*.tiff')))\n        tumor    = len(list((root / 'data' / 'camelyon16' / 'tumor').glob('*.tif'))) + len(list((root / 'data' / 'camelyon16' / 'tumor').glob('*.tiff')))\n        score = features + normal + tumor\n        if score > best_score:\n            best_root, best_score = root, score\n    return best_root\n\nROOT = _pick_project_root()\nFEATURES_DIR = ROOT / 'data' / 'features'\nNORMAL_DIR   = ROOT / 'data' / 'camelyon16' / 'normal'\nTUMOR_DIR    = ROOT / 'data' / 'camelyon16' / 'tumor'\n\nprint(f'Using data root: {ROOT / "data"}')\n

Device: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
VRAM: 8.1 GB
Feature files: 0
Normal slides: 0
Tumor slides:  0


## Cell 2 — Model Definition

In [2]:
# Smoke test
fake = Data(x=torch.randn(40,2048), edge_index=torch.randint(0,40,(2,160)),
            y=torch.tensor([1]), batch=torch.zeros(40,dtype=torch.long)).to(DEVICE)
with torch.no_grad():
    lgt, att = model(fake)
print(f'Smoke test OK — logits {lgt.shape}, attn {att.shape}')

NameError: name 'model' is not defined

## Cell 3 — Training & Evaluation Functions

In [4]:
def train_epoch(model, loader, opt, device):
    model.train(); total, n = 0.0, 0
    for batch in loader:
        batch = batch.to(device); opt.zero_grad()
        logits, _ = model(batch)
        loss = F.cross_entropy(logits, batch.y.squeeze())
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); total += loss.item(); n += 1
    return total / max(n, 1)

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_probs, all_labels, total_loss, n = [], [], 0.0, 0
    for batch in loader:
        batch = batch.to(device)
        logits, _ = model(batch)
        total_loss += F.cross_entropy(logits, batch.y.squeeze()).item()
        all_probs.extend(torch.softmax(logits,1)[:,1].cpu().tolist())
        all_labels.extend(batch.y.squeeze().cpu().tolist())
        n += 1
    p, l = np.array(all_probs), np.array(all_labels)
    preds = (p >= 0.5).astype(int)
    auc  = roc_auc_score(l, p) if len(np.unique(l)) > 1 else 0.5
    f1   = f1_score(l, preds, zero_division=0)
    tp   = int(((preds==1)&(l==1)).sum()); fn = int(((preds==0)&(l==1)).sum())
    tn   = int(((preds==0)&(l==0)).sum()); fp = int(((preds==1)&(l==0)).sum())
    return {'loss': total_loss/max(n,1), 'auc': auc, 'f1': f1,
            'sensitivity': tp/max(tp+fn,1), 'specificity': tn/max(tn+fp,1),
            'probs': p, 'labels': l}


def train_full(model, train_loader, val_loader, device,
               epochs=50, lr=1e-4, save_path=None,
               run_name='run', use_wandb=True):
    opt   = torch.optim.Adam(filter(lambda p:p.requires_grad, model.parameters()),
                              lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-6)
    if use_wandb:
        wandb.init(project='PATHQ', name=run_name,
                   config={'epochs':epochs,'lr':lr,'model':run_name})
    best_auc, no_imp = 0.0, 0
    history = {'train_loss':[], 'val_auc':[], 'val_f1':[]}
    for ep in range(1, epochs+1):
        tl  = train_epoch(model, train_loader, opt, device)
        vm  = evaluate(model, val_loader, device)
        sched.step()
        history['train_loss'].append(tl)
        history['val_auc'].append(vm['auc'])
        history['val_f1'].append(vm['f1'])
        if use_wandb:
            wandb.log({'epoch':ep,'train/loss':tl,'val/auc':vm['auc'],
                       'val/f1':vm['f1'],'val/sensitivity':vm['sensitivity']})
        if True: #ep % 5 == 0 or ep == 1:
            print(f'Ep {ep:3d}  loss={tl:.4f}  auc={vm["auc"]:.4f}  f1={vm["f1"]:.4f}')
        if vm['auc'] > best_auc:
            best_auc = vm['auc']; no_imp = 0
            if save_path:
                torch.save({'epoch':ep,'model_state':model.state_dict(),'val_auc':best_auc}, save_path)
        else:
            no_imp += 1
            if no_imp >= 15: print(f'Early stop at ep {ep}'); break
    if use_wandb: wandb.finish()
    print(f'\nBest val AUC: {best_auc:.4f}')
    return best_auc, history

print('Training functions ready.')

Training functions ready.


## Cell 4 — Build PatchCamelyon MIL bags

In [5]:
import timm
import time
from datasets import load_dataset
from torchvision import transforms

BAG_SIZE = 32

print('Loading PatchCamelyon...')
pcam = load_dataset('1aurent/PatchCamelyon')
print(f'Available splits: {list(pcam.keys())}')

# ── Create validation split from train ─────────────────────────────────────
if 'validation' not in pcam:
    print('Creating validation split (10% of train)...')
    train_val = pcam['train'].train_test_split(test_size=0.1, seed=42)
    pcam['train']      = train_val['train']
    pcam['validation'] = train_val['test']

print(f'  Train      : {len(pcam["train"]):,}')
print(f'  Validation : {len(pcam["validation"]):,}')
print(f'  Test       : {len(pcam["test"]):,}')

# ── Frozen ResNet-50 feature extractor ─────────────────────────────────────
print('\nLoading ResNet-50 backbone...')
backbone  = timm.create_model('resnet50', pretrained=True, num_classes=0)
extractor = backbone.to(DEVICE).eval()
for p in extractor.parameters():
    p.requires_grad = False
print(f'Extractor on: {next(extractor.parameters()).device}')

tfm = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# ── Graph builder ───────────────────────────────────────────────────────────
def make_graphs(split, n_bags=500):
    ds   = pcam[split]
    idxs = np.random.permutation(len(ds))[:n_bags * BAG_SIZE]
    graphs = []

    for i in range(0, len(idxs), BAG_SIZE):
        bi = idxs[i:i + BAG_SIZE]
        if len(bi) < BAG_SIZE:
            continue

        imgs   = torch.stack([tfm(ds[int(j)]['image']) for j in bi]).to(DEVICE)
        labels = [ds[int(j)]['label'] for j in bi]
        bag_lbl = 1 if any(labels) else 0

        with torch.no_grad():
            feats = extractor(imgs).cpu()

        side   = int(BAG_SIZE ** 0.5)
        coords = torch.tensor(
            [(r, c) for r in range(side) for c in range(side)],
            dtype=torch.float32
        )[:BAG_SIZE]

        tree = KDTree(coords.numpy())
        _, nn_idx = tree.query(coords.numpy(), k=min(5, BAG_SIZE))

        src, tgt = [], []
        for ni, nbrs in enumerate(nn_idx):
            for nj in nbrs[1:]:
                src += [ni, int(nj)]
                tgt += [int(nj), ni]

        graphs.append(Data(
            x          = feats,
            coords     = coords,
            edge_index = torch.tensor([src, tgt], dtype=torch.long),
            y          = torch.tensor([bag_lbl],  dtype=torch.long)
        ))

    pos = sum(g.y.item() for g in graphs)
    neg = len(graphs) - pos
    bal = pos / max(len(graphs), 1) * 100
    print(f'  {split:12s}: {len(graphs)} bags  '
          f'pos={pos}  neg={neg}  balance={bal:.1f}%')
    return graphs

# ── Build all splits ────────────────────────────────────────────────────────
print('\nBuilding MIL bags (takes ~3-5 min)...')
t0 = time.time()

train_g = make_graphs('train',      600)
val_g   = make_graphs('validation', 200)
test_g  = make_graphs('test',       200)

print(f'Graph building done in {(time.time()-t0)/60:.1f} min')

# ── DataLoaders ─────────────────────────────────────────────────────────────
tr_ldr = PyGLoader(train_g, batch_size=8, shuffle=True)
va_ldr = PyGLoader(val_g,   batch_size=8, shuffle=False)
te_ldr = PyGLoader(test_g,  batch_size=8, shuffle=False)

print(f'\nLoaders ready.')
print(f'  Train batches : {len(tr_ldr)}')
print(f'  Val batches   : {len(va_ldr)}')
print(f'  Test batches  : {len(te_ldr)}')

Loading PatchCamelyon...
Available splits: ['train', 'valid', 'test']
Creating validation split (10% of train)...
  Train      : 235,929
  Validation : 26,215
  Test       : 32,768

Loading ResNet-50 backbone...
Extractor on: cuda:0

Building MIL bags (takes ~3-5 min)...
  train       : 600 bags  pos=600  neg=0  balance=100.0%
  validation  : 200 bags  pos=200  neg=0  balance=100.0%
  test        : 200 bags  pos=200  neg=0  balance=100.0%
Graph building done in 0.5 min

Loaders ready.
  Train batches : 75
  Val batches   : 25
  Test batches  : 25


## Cell 5 — Train on PatchCamelyon

In [ ]:
import time

print('Training on PatchCamelyon — target AUC > 0.90')
print('='*55)

# ── Confirm device ──────────────────────────────────────────
print(f'Device     : {DEVICE}')
print(f'CUDA avail : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU        : {torch.cuda.get_device_name(0)}')
    print(f'VRAM       : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print('='*55)

model_pcam = ClassicalGNN().to(DEVICE)
n_p = sum(p.numel() for p in model_pcam.parameters() if p.requires_grad)
print(f'Model params: {n_p:,}')

t0 = time.time()
best_auc_pcam, hist = train_full(
    model_pcam, tr_ldr, va_ldr, DEVICE,
    epochs=40, lr=1e-4,
    save_path=CKPT_DIR/'classical_pcam_best.pth',
    run_name='classical_pcam',
    use_wandb=False                               # ← no wandb hang
)
print(f'\nTotal training time: {(time.time()-t0)/60:.1f} minutes')

# ── Load best and evaluate ──────────────────────────────────
print('\nLoading best checkpoint...')
ckpt = torch.load(CKPT_DIR/'classical_pcam_best.pth', weights_only=False)
model_pcam.load_state_dict(ckpt['model_state'])
print(f'Best checkpoint: epoch {ckpt["epoch"]}  val AUC {ckpt["val_auc"]:.4f}')

tm = evaluate(model_pcam, te_ldr, DEVICE)

print('\n=== PatchCamelyon TEST RESULTS ===')
for k, v in tm.items():
    if k not in ('probs', 'labels'):
        print(f'  {k:15s}: {v:.4f}')

PCAM_AUC = tm['auc']
if   PCAM_AUC > 0.90: status = ' PASS'
elif PCAM_AUC > 0.85: status = '  CLOSE — run more epochs'
else:                  status = ' Need more epochs / check GPU'

print(f'\nTarget: > 0.90  |  Got: {PCAM_AUC:.4f}  |  {status}')

In [6]:
import timm
import time
import numpy as np
import torch
import torch.nn.functional as F
from datasets import load_dataset
from torchvision import transforms
from sklearn.metrics import roc_auc_score, f1_score

# ── Config ──────────────────────────────────────────────────────────────────
BAG_SIZE   = 32
N_TRAIN    = 600
N_VAL      = 200
N_TEST     = 200
EPOCHS     = 40
LR         = 1e-4
BATCH_SIZE = 8

# ── Device ──────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('='*60)
print(f'Device     : {DEVICE}')
print(f'CUDA avail : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU        : {torch.cuda.get_device_name(0)}')
    print(f'VRAM       : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print('='*60)

# ════════════════════════════════════════════════════════════
# 1. LOAD DATASET
# ════════════════════════════════════════════════════════════
print('\nLoading PatchCamelyon...')
pcam = load_dataset('1aurent/PatchCamelyon')
print(f'Available splits: {list(pcam.keys())}')

if 'validation' not in pcam:
    print('Creating validation split (10% of train)...')
    train_val          = pcam['train'].train_test_split(test_size=0.1, seed=42)
    pcam['train']      = train_val['train']
    pcam['validation'] = train_val['test']

print(f'  Train      : {len(pcam["train"]):,}')
print(f'  Validation : {len(pcam["validation"]):,}')
print(f'  Test       : {len(pcam["test"]):,}')

# ════════════════════════════════════════════════════════════
# 2. FROZEN ResNet-50 FEATURE EXTRACTOR
# ════════════════════════════════════════════════════════════
print('\nLoading ResNet-50 backbone (frozen)...')
backbone  = timm.create_model('resnet50', pretrained=True, num_classes=0)
extractor = backbone.to(DEVICE).eval()
for p in extractor.parameters():
    p.requires_grad = False
print(f'Extractor on : {next(extractor.parameters()).device}')

tfm = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# ════════════════════════════════════════════════════════════
# 3. GRAPH BUILDER
# ════════════════════════════════════════════════════════════
def make_graphs(split, n_bags=500):
    ds    = pcam[split]
    idxs  = np.random.permutation(len(ds))[:n_bags * BAG_SIZE]
    graphs = []

    for i in range(0, len(idxs), BAG_SIZE):
        bi = idxs[i:i + BAG_SIZE]
        if len(bi) < BAG_SIZE:
            continue

        imgs    = torch.stack([tfm(ds[int(j)]['image']) for j in bi]).to(DEVICE)
        labels  = [ds[int(j)]['label'] for j in bi]
        bag_lbl = 1 if any(labels) else 0

        with torch.no_grad():
            feats = extractor(imgs).cpu()

        side   = int(BAG_SIZE ** 0.5)
        coords = torch.tensor(
            [(r, c) for r in range(side) for c in range(side)],
            dtype=torch.float32
        )[:BAG_SIZE]

        tree = KDTree(coords.numpy())
        _, nn_idx = tree.query(coords.numpy(), k=min(5, BAG_SIZE))

        src, tgt = [], []
        for ni, nbrs in enumerate(nn_idx):
            for nj in nbrs[1:]:
                src += [ni, int(nj)]
                tgt += [int(nj), ni]

        graphs.append(Data(
            x          = feats,
            coords     = coords,
            edge_index = torch.tensor([src, tgt], dtype=torch.long),
            y          = torch.tensor([bag_lbl],  dtype=torch.long)
        ))

    pos = sum(g.y.item() for g in graphs)
    neg = len(graphs) - pos
    bal = pos / max(len(graphs), 1) * 100
    print(f'  {split:12s}: {len(graphs)} bags  '
          f'pos={pos}  neg={neg}  balance={bal:.1f}%')
    return graphs

print('\nBuilding MIL bags...')
t0      = time.time()
train_g = make_graphs('train',      N_TRAIN)
val_g   = make_graphs('validation', N_VAL)
test_g  = make_graphs('test',       N_TEST)
print(f'Graph building done in {(time.time()-t0)/60:.1f} min')

tr_ldr = PyGLoader(train_g, batch_size=BATCH_SIZE, shuffle=True)
va_ldr = PyGLoader(val_g,   batch_size=BATCH_SIZE, shuffle=False)
te_ldr = PyGLoader(test_g,  batch_size=BATCH_SIZE, shuffle=False)

print(f'\nLoaders ready.')
print(f'  Train batches : {len(tr_ldr)}')
print(f'  Val batches   : {len(va_ldr)}')
print(f'  Test batches  : {len(te_ldr)}')

# ════════════════════════════════════════════════════════════
# 4. TRAIN / EVAL FUNCTIONS
# ════════════════════════════════════════════════════════════
def train_epoch(model, loader, opt, device):
    model.train()
    total, n = 0.0, 0
    for batch in loader:
        batch = batch.to(device)
        opt.zero_grad()
        logits, _ = model(batch)
        loss = F.cross_entropy(logits, batch.y.squeeze())
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        total += loss.item()
        n     += 1
    return total / max(n, 1)

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_probs, all_labels, total_loss, n = [], [], 0.0, 0
    for batch in loader:
        batch = batch.to(device)
        logits, _ = model(batch)
        total_loss += F.cross_entropy(logits, batch.y.squeeze()).item()
        all_probs.extend(torch.softmax(logits, 1)[:, 1].cpu().tolist())
        all_labels.extend(batch.y.squeeze().cpu().tolist())
        n += 1

    p, l  = np.array(all_probs), np.array(all_labels)
    preds = (p >= 0.5).astype(int)
    auc   = roc_auc_score(l, p) if len(np.unique(l)) > 1 else 0.5
    f1    = f1_score(l, preds, zero_division=0)
    tp    = int(((preds==1) & (l==1)).sum())
    fn    = int(((preds==0) & (l==1)).sum())
    tn    = int(((preds==0) & (l==0)).sum())
    fp    = int(((preds==1) & (l==0)).sum())

    return {
        'loss':        total_loss / max(n, 1),
        'auc':         auc,
        'f1':          f1,
        'sensitivity': tp / max(tp + fn, 1),
        'specificity': tn / max(tn + fp, 1),
        'probs':       p,
        'labels':      l
    }

def train_full(model, train_loader, val_loader, device,
               epochs=50, lr=1e-4, save_path=None, run_name='run'):

    opt   = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=1e-4
    )
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=epochs, eta_min=1e-6
    )

    best_auc, no_imp = 0.0, 0
    history = {'train_loss': [], 'val_auc': [], 'val_f1': []}

    print(f'\n{"Ep":>4} {"Loss":>8} {"AUC":>8} {"F1":>7} '
          f'{"Sens":>7} {"Spec":>7} {"Time":>7}  Best')
    print('─' * 65)

    for ep in range(1, epochs + 1):
        ep_start = time.time()

        tl = train_epoch(model, train_loader, opt, device)
        vm = evaluate(model, val_loader, device)
        sched.step()

        ep_time = time.time() - ep_start

        history['train_loss'].append(tl)
        history['val_auc'].append(vm['auc'])
        history['val_f1'].append(vm['f1'])

        improved = vm['auc'] > best_auc
        marker   = '✓' if improved else ''

        print(f'{ep:4d} {tl:8.4f} {vm["auc"]:8.4f} {vm["f1"]:7.4f} '
              f'{vm["sensitivity"]:7.4f} {vm["specificity"]:7.4f} '
              f'{ep_time:6.1f}s  {marker}')

        if improved:
            best_auc = vm['auc']
            no_imp   = 0
            if save_path:
                torch.save({
                    'epoch':       ep,
                    'model_state': model.state_dict(),
                    'val_auc':     best_auc
                }, save_path)
        else:
            no_imp += 1
            if no_imp >= 15:
                print(f'\nEarly stop at epoch {ep} — no improvement for 15 epochs')
                break

    print('─' * 65)
    print(f'Best val AUC : {best_auc:.4f}')
    return best_auc, history

# ════════════════════════════════════════════════════════════
# 5. TRAIN
# ════════════════════════════════════════════════════════════
print('\n' + '='*60)
print('Training ClassicalGNN on PatchCamelyon — target AUC > 0.90')
print('='*60)

model_pcam = ClassicalGNN().to(DEVICE)
n_p = sum(p.numel() for p in model_pcam.parameters() if p.requires_grad)
print(f'Trainable params : {n_p:,}')

t_start = time.time()

best_auc_pcam, hist = train_full(
    model_pcam, tr_ldr, va_ldr, DEVICE,
    epochs    = EPOCHS,
    lr        = LR,
    save_path = CKPT_DIR / 'classical_pcam_best.pth',
    run_name  = 'classical_pcam'
)

print(f'\nTotal training time : {(time.time()-t_start)/60:.1f} minutes')

# ════════════════════════════════════════════════════════════
# 6. TEST
# ════════════════════════════════════════════════════════════
print('\nLoading best checkpoint...')
ckpt = torch.load(CKPT_DIR / 'classical_pcam_best.pth', weights_only=False)
model_pcam.load_state_dict(ckpt['model_state'])
print(f'Best checkpoint  : epoch {ckpt["epoch"]}  val AUC {ckpt["val_auc"]:.4f}')

tm = evaluate(model_pcam, te_ldr, DEVICE)

print('\n' + '='*40)
print('   PatchCamelyon TEST RESULTS')
print('='*40)
for k, v in tm.items():
    if k not in ('probs', 'labels'):
        print(f'  {k:15s}: {v:.4f}')

PCAM_AUC = tm['auc']
if   PCAM_AUC > 0.90: status = ''
elif PCAM_AUC > 0.85: status = '️  CLOSE — try more epochs'
else:                  status = ' Need more epochs / check GPU'

print(f'\nTarget : > 0.90')
print(f'Got    : {PCAM_AUC:.4f}')
print(f'Status : {status}')

Device     : cuda
CUDA avail : True
GPU        : NVIDIA GeForce RTX 5060 Laptop GPU
VRAM       : 8.1 GB

Loading PatchCamelyon...
Available splits: ['train', 'valid', 'test']
Creating validation split (10% of train)...
  Train      : 235,929
  Validation : 26,215
  Test       : 32,768

Loading ResNet-50 backbone (frozen)...
Extractor on : cuda:0

Building MIL bags...
  train       : 600 bags  pos=600  neg=0  balance=100.0%
  validation  : 200 bags  pos=200  neg=0  balance=100.0%
  test        : 200 bags  pos=200  neg=0  balance=100.0%
Graph building done in 0.5 min

Loaders ready.
  Train batches : 75
  Val batches   : 25
  Test batches  : 25

Training ClassicalGNN on PatchCamelyon — target AUC > 0.90
Trainable params : 346,946

  Ep     Loss      AUC      F1    Sens    Spec    Time  Best
─────────────────────────────────────────────────────────────────


RuntimeError: mat1 and mat2 shapes cannot be multiplied (256x2048 and 512x256)

## Cell 6 — Training Curves Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist['train_loss'], color='#B83A12', label='Train Loss')
axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(hist['val_auc'], color='#0A6B6B', label='Val AUC')
axes[1].plot(hist['val_f1'],  color='#534AB7', label='Val F1')
axes[1].axhline(0.90, color='red', ls='--', alpha=0.5, label='AUC target')
axes[1].set_title('Validation Metrics'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(str(OUT_DIR/'week3_training_curves.png'), dpi=120, bbox_inches='tight')
plt.show()

## Cell 7 — CAMELYON16 Training (if features exist)

In [ ]:
sys.path.insert(0, str(ROOT))
from pathq.dataset import get_loaders

n_feat = len(list(FEATURES_DIR.glob('*_features.pt')))
print(f'Feature files found: {n_feat}')

CAMELYON_AUC = None

if n_feat < 5:
    print('Not enough features yet — complete Week 2 extraction first.')
    print(f'Using PatchCamelyon AUC={PCAM_AUC:.4f} as baseline reference.')
else:
    tr, va, te = get_loaders(FEATURES_DIR, NORMAL_DIR, TUMOR_DIR, batch_size=4)
    model_cam = ClassicalGNN().to(DEVICE)
    best_auc_cam, hist_cam = train_full(
        model_cam, tr, va, DEVICE, epochs=50, lr=1e-4,
        save_path=CKPT_DIR/'classical_cam16_best.pth',
        run_name='classical_cam16'
    )
    ckpt = torch.load(CKPT_DIR/'classical_cam16_best.pth', weights_only=False)
    model_cam.load_state_dict(ckpt['model_state'])
    cm = evaluate(model_cam, te, DEVICE)
    CAMELYON_AUC = cm['auc']
    print('\n=== CAMELYON16 TEST (Classical Baseline — Paper Table 1 Row 1) ===')
    for k,v in cm.items():
        if k not in ('probs','labels'): print(f'  {k:15s}: {v:.4f}')

# Save baseline results
results = {'pcam_classical_auc': PCAM_AUC,
           'cam16_classical_auc': CAMELYON_AUC}
with open(OUT_DIR/'baseline_results.json','w') as f:
    json.dump(results, f, indent=2)
print('\nSaved: outputs/baseline_results.json')
print('These numbers go in your paper Table 1 Row 1 (Classical Baseline).')

## Cell 8 — Attention Map Visualisation

In [ ]:
model_pcam.eval()
samples = test_g[:4]
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
fig.suptitle('ABMIL Attention Weights — Layer 2 XAI', fontsize=13)

for i, graph in enumerate(samples):
    g = graph.to(DEVICE)
    g.batch = torch.zeros(g.num_nodes, dtype=torch.long, device=DEVICE)
    with torch.no_grad():
        logits, attn = model_pcam(g)
    pred  = torch.softmax(logits,1)[0,1].item()
    attn_np = attn.squeeze(1).cpu().numpy()
    coords  = g.coords.cpu().numpy()
    true_l  = graph.y.item()

    sc = axes[0][i].scatter(coords[:,1], coords[:,0],
                            c=attn_np, cmap='hot', s=100)
    axes[0][i].set_title(
        f'True: {"TUM" if true_l else "NOR"}  Pred: {pred:.2f}',
        fontsize=9, color='red' if true_l else 'green')
    axes[0][i].invert_yaxis(); axes[0][i].axis('off')
    plt.colorbar(sc, ax=axes[0][i], fraction=0.04)

    axes[1][i].bar(range(len(attn_np)), np.sort(attn_np)[::-1],
                   color='#B83A12', alpha=0.8)
    axes[1][i].set_xlabel('Patch rank (sorted)'); axes[1][i].set_ylabel('Weight')

plt.tight_layout()
plt.savefig(str(OUT_DIR/'week3_attention_maps.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Attention map saved.')

In [ ]:
print('WEEK 3 COMPLETE')
print('='*50)
print(f'PatchCamelyon AUC (classical):  {PCAM_AUC:.4f}')
if CAMELYON_AUC: print(f'CAMELYON16 AUC (classical):     {CAMELYON_AUC:.4f}')
print()
print('Saved: outputs/baseline_results.json  <- your paper Table 1 Row 1')
print('Next:  week4_vqc_prototype.ipynb')